In [2]:
import signal
import wandb
import torch
import os 

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from hydra import compose, initialize

from codefiles.helpers import is_running_in_notebook  # for reloading modules instead of restarting kernel
if is_running_in_notebook():
    from codefiles import helpers
    import importlib
    importlib.reload(helpers)
from codefiles.helpers import set_all_seeds, signal_handler, build_model, build_lightningmodule, build_datamodule

os.environ["WANDB_SILENT"] = "true"
torch.set_float32_matmul_precision("high")

def main(cfg) -> None:
    wandb.finish()
    set_all_seeds(seed=cfg.seed)
    wandb.init(
        project=cfg.wandb.project,
        group=None if cfg.wandb.group == "None" else cfg.wandb.group,
        config={key: value for key, value in cfg.items()},
    )

    model = build_model(cfg)
    lightningmodule = build_lightningmodule(cfg, model)
    datamodule = build_datamodule(cfg)

    trainer = pl.Trainer(
        logger=WandbLogger(project=cfg.wandb.project, dir="wandb/"),
        log_every_n_steps=1,
        accelerator='gpu',
        devices=1,
        max_epochs=cfg.max_epochs,
        precision=cfg.precision
    )

    trainer.fit(lightningmodule, datamodule)
    wandb.finish()

if __name__ == "__main__":
    CONFIG_NAME = "config"
    signal.signal(signal.SIGINT, signal_handler)
    with initialize(version_base="1.1", config_path="config"):
        cfg = compose(config_name=f"{CONFIG_NAME}")
    main(cfg)

Seed set to 420
/sc-projects/sc-proj-ukb-cvd/environments/mml/lib/python3.9/site-packages/pytorch_lightning/utilities/parsing.py:208: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.
/sc-projects/sc-proj-ukb-cvd/environments/mml/lib/python3.9/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /sc-projects/sc-proj-ukb-cvd/environments/mml/lib/py ...
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/sc-projects/sc-proj-ukb-cvd/environments/mml/lib/python3.9/site-packages/pytorch_lightning/loggers/wandb.py:396: There is a wandb run already in progress and newly

total_samples: 27471 / 27471
no_missing: 27471 / 27471
1_missing: 0 / 27471
modality_0_missing: 0 / 27471
modality_1_missing: 0 / 27471
total_samples: 8755 / 8755
no_missing: 8755 / 8755
1_missing: 0 / 8755
modality_0_missing: 0 / 8755
modality_1_missing: 0 / 8755


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


total_samples: 8824 / 8824
no_missing: 8824 / 8824
1_missing: 0 / 8824
modality_0_missing: 0 / 8824
modality_1_missing: 0 / 8824



  | Name               | Type                         | Params | Mode 
----------------------------------------------------------------------------
0 | model              | Multimodal_Architecture      | 99.9 M | train
1 | loss               | WeightedNaNBCEWithLogitsLoss | 0      | train
2 | acc_train          | MulticlassAccuracy           | 0      | train
3 | metric_train_macro | NaNMultilabelAUROC           | 0      | train
4 | metric_val_macro   | NaNMultilabelAUROC           | 0      | train
5 | metric_test_macro  | NaNMultilabelAUROC           | 0      | train
6 | metric_train_micro | NaNMultilabelAUROC           | 0      | train
7 | metric_val_micro   | NaNMultilabelAUROC           | 0      | train
8 | metric_test_micro  | NaNMultilabelAUROC           | 0      | train
----------------------------------------------------------------------------
99.9 M    Trainable params
0         Non-trainable params
99.9 M    Total params
399.533   Total estimated model params size (MB)
260  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

KeyError: 'ecg'